# Forecast Granularity Control-Group Comparison

This notebook supplies the missing control group and compares three approaches over the **same final four-week test window**:

1. Daily × SKU direct forecast (`seasonal_naive_7`), then aggregated to SKU-week for comparable evaluation.
2. Weekly × SKU direct forecast (`moving_average_8`).
3. Weekly × Sub-Group forecast (`moving_average_8`) allocated back to SKUs using leakage-free historical shares.

The primary comparison uses **pooled WAPE evaluated at the SKU-week grain** for all three approaches. The earlier daily-SKU median WAPE is also reported separately so that it is not mixed with pooled WAPE.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    (
        path
        for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
        if (path / "config.py").is_file()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find config.py. Start Jupyter from the repository or its Notebooks folder."
    )
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd

from config import (
    DAILY_FLOW_FILE,
    GRANULARITY_FILE,
    HOLDOUT_WEEKS,
    PROCESSED_DIR,
    RECENT_SHARE_WEIGHT,
    RECENT_WEEKS,
    WEEK_FREQUENCY,
    ensure_output_directories,
)

INPUT_FILE = DAILY_FLOW_FILE
OUTPUT_DIR = PROCESSED_DIR
TEST_WEEKS = HOLDOUT_WEEKS
RECENT_SHARE_WEEKS = RECENT_WEEKS

ensure_output_directories()
df = pd.read_csv(INPUT_FILE)
df["Date"] = pd.to_datetime(df["Date"])

required = {"Date", "product_id", "Group", "Sub-Group", "sales_units"}
missing = required.difference(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")
if df.duplicated(["Date", "product_id"]).any():
    raise ValueError("Duplicate Date + product_id keys found")
if (df["sales_units"] < 0).any():
    raise ValueError("Negative sales_units found")

# Use full seven-day weeks ending Wednesday; this preserves the complete final week.
df["week_end"] = df["Date"].dt.to_period(WEEK_FREQUENCY).dt.end_time.dt.normalize()
week_day_counts = df.groupby("week_end")["Date"].nunique()
complete_weeks = week_day_counts[week_day_counts == 7].index
df_complete = df[df["week_end"].isin(complete_weeks)].copy()

weekly_sku = (
    df_complete.groupby(
        ["week_end", "Group", "Sub-Group", "product_id"], as_index=False
    )["sales_units"].sum()
)

all_weeks = sorted(weekly_sku["week_end"].unique())
test_weeks = all_weeks[-TEST_WEEKS:]
train_weeks = all_weeks[:-TEST_WEEKS]
test_start = pd.Timestamp(test_weeks[0]) - pd.Timedelta(days=6)
test_end = pd.Timestamp(test_weeks[-1])

train_daily = df[df["Date"] < test_start].copy()
test_daily = df[df["Date"].between(test_start, test_end)].copy()
train_weekly = weekly_sku[weekly_sku["week_end"].isin(train_weeks)].copy()
test_weekly = weekly_sku[weekly_sku["week_end"].isin(test_weeks)].copy()

print("Test window:", test_start.date(), "to", test_end.date())
print("Test weeks:", pd.to_datetime(test_weeks).date)
print("Daily test rows:", len(test_daily))
print("Weekly test rows:", len(test_weekly))

## Helper functions

In [ ]:
def pooled_metrics(actual, forecast):
    actual = np.asarray(actual, dtype=float)
    forecast = np.asarray(forecast, dtype=float)
    denominator = actual.sum()
    if denominator <= 0:
        return {"wape": np.nan, "forecast_bias": np.nan}
    return {
        "wape": np.abs(actual - forecast).sum() / denominator,
        "forecast_bias": (forecast.sum() - denominator) / denominator,
    }


def build_sku_shares(historical_sku, recent_weeks=8, recent_weight=0.80):
    product_dim = historical_sku[
        ["Group", "Sub-Group", "product_id"]
    ].drop_duplicates()

    full = (
        historical_sku.groupby(
            ["Group", "Sub-Group", "product_id"], as_index=False
        )["sales_units"].sum()
        .rename(columns={"sales_units": "full_sales"})
    )
    recent_week_values = sorted(historical_sku["week_end"].unique())[-recent_weeks:]
    recent = (
        historical_sku[historical_sku["week_end"].isin(recent_week_values)]
        .groupby(["Group", "Sub-Group", "product_id"], as_index=False)[
            "sales_units"
        ].sum()
        .rename(columns={"sales_units": "recent_sales"})
    )

    shares = (
        product_dim
        .merge(full, on=["Group", "Sub-Group", "product_id"], how="left")
        .merge(recent, on=["Group", "Sub-Group", "product_id"], how="left")
    )
    shares[["full_sales", "recent_sales"]] = shares[
        ["full_sales", "recent_sales"]
    ].fillna(0)
    keys = ["Group", "Sub-Group"]
    shares["full_total"] = shares.groupby(keys)["full_sales"].transform("sum")
    shares["recent_total"] = shares.groupby(keys)["recent_sales"].transform("sum")
    shares["sku_count"] = shares.groupby(keys)["product_id"].transform("count")

    shares["full_share"] = np.where(
        shares["full_total"] > 0,
        shares["full_sales"] / shares["full_total"],
        1 / shares["sku_count"],
    )
    shares["recent_share"] = np.where(
        shares["recent_total"] > 0,
        shares["recent_sales"] / shares["recent_total"],
        shares["full_share"],
    )
    shares["allocation_share"] = (
        recent_weight * shares["recent_share"]
        + (1 - recent_weight) * shares["full_share"]
    )
    shares["allocation_share"] = shares["allocation_share"] / shares.groupby(
        keys
    )["allocation_share"].transform("sum")
    return shares[["Group", "Sub-Group", "product_id", "allocation_share"]]


## 1. Daily × SKU direct forecast

In [ ]:
daily_rows = []
daily_product_wape = []

for product_id in sorted(df["product_id"].unique()):
    history = (
        train_daily.loc[train_daily["product_id"] == product_id]
        .sort_values("Date")["sales_units"].to_numpy()
    )
    actual_frame = (
        test_daily.loc[
            test_daily["product_id"] == product_id,
            ["Date", "week_end", "Group", "Sub-Group", "product_id", "sales_units"],
        ].sort_values("Date")
    )
    actual = actual_frame["sales_units"].to_numpy()
    prediction = np.resize(history[-7:], len(actual))

    product_metric = pooled_metrics(actual, prediction)
    daily_product_wape.append({
        "product_id": product_id,
        "wape": product_metric["wape"],
    })

    output = actual_frame.copy()
    output["forecast_units"] = prediction
    daily_rows.append(output)

daily_predictions = pd.concat(daily_rows, ignore_index=True)
daily_original = pooled_metrics(
    daily_predictions["sales_units"], daily_predictions["forecast_units"]
)

# Aggregate the daily model's predictions to SKU-week before the primary comparison.
daily_as_weekly = (
    daily_predictions.groupby(
        ["week_end", "Group", "Sub-Group", "product_id"], as_index=False
    ).agg(
        actual_units=("sales_units", "sum"),
        forecast_units=("forecast_units", "sum"),
    )
)
daily_weekly_metric = pooled_metrics(
    daily_as_weekly["actual_units"], daily_as_weekly["forecast_units"]
)
daily_median_sku_wape = pd.DataFrame(daily_product_wape)["wape"].median()

print("Original daily pooled WAPE:", daily_original["wape"])
print("Original median SKU WAPE:", daily_median_sku_wape)
print("Daily model evaluated at SKU-week pooled WAPE:", daily_weekly_metric["wape"])

## 2. Weekly × SKU direct moving-average-8 control group

In [ ]:
weekly_direct_rows = []

for keys, history_frame in train_weekly.groupby(
    ["Group", "Sub-Group", "product_id"]
):
    group_name, subgroup_name, product_id = keys
    history = history_frame.sort_values("week_end")["sales_units"].to_numpy()
    forecast_value = history[-min(8, len(history)):].mean()

    actual_frame = (
        test_weekly.loc[
            (test_weekly["Group"] == group_name)
            & (test_weekly["Sub-Group"] == subgroup_name)
            & (test_weekly["product_id"] == product_id)
        ]
        .sort_values("week_end")
        .copy()
    )
    actual_frame["forecast_units"] = forecast_value
    weekly_direct_rows.append(actual_frame)

weekly_direct = pd.concat(weekly_direct_rows, ignore_index=True)
weekly_direct_metric = pooled_metrics(
    weekly_direct["sales_units"], weekly_direct["forecast_units"]
)

print("Weekly direct SKU MA8 pooled WAPE:", weekly_direct_metric["wape"])
print("Weekly direct SKU MA8 bias:", weekly_direct_metric["forecast_bias"])

## 3. Weekly × Sub-Group MA8 allocated back to SKUs

In [ ]:
train_subgroup = (
    train_weekly.groupby(["week_end", "Group", "Sub-Group"], as_index=False)[
        "sales_units"
    ].sum()
)
shares = build_sku_shares(
    train_weekly,
    recent_weeks=RECENT_SHARE_WEEKS,
    recent_weight=RECENT_SHARE_WEIGHT,
)

subgroup_forecasts = []
for keys, history_frame in train_subgroup.groupby(["Group", "Sub-Group"]):
    group_name, subgroup_name = keys
    history = history_frame.sort_values("week_end")["sales_units"].to_numpy()
    forecast_value = history[-min(8, len(history)):].mean()
    for week_end in test_weeks:
        subgroup_forecasts.append({
            "week_end": pd.Timestamp(week_end),
            "Group": group_name,
            "Sub-Group": subgroup_name,
            "subgroup_forecast_units": forecast_value,
        })

subgroup_forecasts = pd.DataFrame(subgroup_forecasts)
topdown_forecasts = subgroup_forecasts.merge(
    shares, on=["Group", "Sub-Group"], how="left"
)
topdown_forecasts["forecast_units"] = (
    topdown_forecasts["subgroup_forecast_units"]
    * topdown_forecasts["allocation_share"]
)

topdown = test_weekly.merge(
    topdown_forecasts,
    on=["week_end", "Group", "Sub-Group", "product_id"],
    how="left",
    validate="one_to_one",
)
if topdown["forecast_units"].isna().any():
    raise ValueError("Missing forecasts after top-down allocation")

topdown_metric = pooled_metrics(topdown["sales_units"], topdown["forecast_units"])

reconciliation = (
    topdown_forecasts.groupby(["week_end", "Group", "Sub-Group"])[
        "forecast_units"
    ].sum()
    - subgroup_forecasts.set_index(["week_end", "Group", "Sub-Group"])[
        "subgroup_forecast_units"
    ]
).abs().max()

print("Top-down SKU pooled WAPE:", topdown_metric["wape"])
print("Top-down SKU bias:", topdown_metric["forecast_bias"])
print("Maximum reconciliation difference:", reconciliation)

## 4. Clean comparison table

In [ ]:
comparison = pd.DataFrame([
    {
        "approach": "Daily x SKU direct -> aggregate to SKU-week",
        "model": "seasonal_naive_7",
        "evaluation_grain": "SKU-week",
        "pooled_wape": daily_weekly_metric["wape"],
        "forecast_bias": daily_weekly_metric["forecast_bias"],
        "forecast_accuracy": max(0.0, 1 - daily_weekly_metric["wape"]),
    },
    {
        "approach": "Weekly x SKU direct",
        "model": "moving_average_8",
        "evaluation_grain": "SKU-week",
        "pooled_wape": weekly_direct_metric["wape"],
        "forecast_bias": weekly_direct_metric["forecast_bias"],
        "forecast_accuracy": max(0.0, 1 - weekly_direct_metric["wape"]),
    },
    {
        "approach": "Weekly x Sub-Group -> historical-share SKU allocation",
        "model": "moving_average_8 + 80/20 shares",
        "evaluation_grain": "SKU-week",
        "pooled_wape": topdown_metric["wape"],
        "forecast_bias": topdown_metric["forecast_bias"],
        "forecast_accuracy": max(0.0, 1 - topdown_metric["wape"]),
    },
])

legacy_metric_note = pd.DataFrame([{
    "metric": "Original daily direct median SKU WAPE",
    "value": daily_median_sku_wape,
    "note": "This is the approximately 0.66 figure; do not compare it directly with pooled WAPE.",
}])

comparison.to_csv(GRANULARITY_FILE, index=False)
legacy_metric_note.to_csv(OUTPUT_DIR / "forecast_metric_definition_note.csv", index=False)
weekly_direct.to_csv(OUTPUT_DIR / "weekly_sku_direct_ma8_backtest.csv", index=False)

print("Saved outputs to:", OUTPUT_DIR)
print("\nStrictly comparable pooled SKU-week WAPE:")
print(comparison.to_string(index=False))
print("\nLegacy 0.66 metric definition:")
print(legacy_metric_note.to_string(index=False))